In [ ]:
# !pip install crewai langchain-openai python-dotenv
# !pip install --upgrade crewai crewai-tools


[notice] A new release of pip available: 22.3 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_groq import ChatGroq
import os
from crewai import Agent, Task, Crew, Process, LLM
# from langchain.tools import tool
from langchain_openai import ChatOpenAI, AzureChatOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# The previous agent has extracted this from the trade documents.
input_json = {
    "key_data": {
        "vessel_name": "MV Brazil Star",
        "applicant_name": "The American Coffee Roasters Co.",
        "beneficiary_name": "São Paulo Coffee Exports Ltd.",
        "shipment_date": "2025-11-15" # Format: YYYY-MM-DD
    }
}

In [3]:
def check_sanctions_list(entity_name: str) -> str:
    """
    Checks a given entity name against a mock sanctions database API.
    Returns 'Clear' if no match is found, or 'Potential Match' with details.
    """
    print(f"--- Checking sanctions for: {entity_name} ---")
    # In a real application, this would be a live API call.
    # We are mocking the logic for this example.
    if "terror" in entity_name.lower() or "sanctioned" in entity_name.lower():
        return f"Potential Match: {entity_name} found on sanctions list."
    return f"Clear: {entity_name} not found on sanctions list."

In [4]:
import httpx
client = httpx.Client(verify=False)

In [7]:
import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ['LITELLM_DISABLE_SSL'] = 'True'
print(certifi.where())

C:\Users\UN177KK\AppData\Local\.certifi\cacert.pem


In [ ]:
# llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.5, http_client=client)
# llm.invoke("what is Trade Finance?")

AIMessage(content="Trade finance is a financial instrument that enables companies to facilitate international trade by providing liquidity and mitigating risks associated with exporting and importing goods. It involves a range of financial products and services that help bridge the gap between a buyer's payment terms and a seller's need for immediate payment.\n\nTrade finance typically involves a combination of the following components:\n\n1. **Payment terms**: The buyer and seller agree on payment terms, such as letter of credit, bill of exchange, or open account.\n2. **Risk management**: Trade finance providers help mitigate risks such as non-payment, currency fluctuations, and cargo damage.\n3. **Financing**: Trade finance providers offer financing options to help the buyer pay for the goods or to help the seller receive payment earlier.\n4. **Insurance**: Trade finance providers may offer insurance products to protect against cargo damage, loss, or other risks.\n\nCommon trade fina

In [10]:
from crewai import LLM
import httpx
import litellm
litellm.client_session = httpx.Client(verify=False)
os.environ["OTEL_SDK_DISABLED"] = "true"

llm = LLM(
    model="groq/llama-3.3-70b-versatile",  # Adjust provider as appropriate
    temperature=0.1,
)
response = llm.call("What is trade finance?")
print(response)

Trade finance is the financial instruments and services that facilitate international trade and commerce by mitigating the risks associated with buying and selling goods and services across borders. It involves the financing of trade transactions, managing the risks of payment, and ensuring the smooth flow of goods and services between buyers and sellers.

Trade finance typically involves a range of financial products and services, including:

1. **Letters of Credit (LCs)**: A letter of credit is a guarantee issued by a bank on behalf of the buyer, promising to pay the seller upon presentation of compliant documents.
2. **Bill of Exchange**: A bill of exchange is a document that requires the buyer to pay the seller at a specified future date.
3. **Factoring**: Factoring involves the purchase of accounts receivable from a seller, allowing the seller to receive immediate payment.
4. **Forfaiting**: Forfaiting involves the purchase of a seller's receivables at a discount, without recourse

In [ ]:
from crewai import Agent


check_sanctions_tool = {
    "name": "check_sanctions_list",
    "description": "Checks a given entity name against a mock sanctions database API.",
    "func": check_sanctions_list
}

compliance_officer = Agent(
    role="Trade Finance Compliance and Regulatory Officer",
    goal="""Ensure a trade finance transaction strictly adheres to international regulations (UCP 600) and passes all AML/KYC sanctions screenings based on the provided JSON data.""",
    backstory="""You are a certified compliance professional specializing in international trade. Your task is to analyze extracted data, not the raw documents, 
    and provide a clear compliance verdict.""",
    # tools=[check_sanctions_tool],
    llm=llm,
    verbose=True,
    # allow_delegation=False
)

# The task description tells the agent exactly what to do with the input JSON.

compliance_task = Task(
    description=f"""
        Analyze the provided JSON data and perform a full compliance review.
        The current date is August 22, 2025.

        Your task has two parts:
        1. **UCP 600 Check**: The JSON contains a 'shipment_date'. According to UCP 600, documents must be presented within 21 days of this date. However, the LC expires on November 30, 2025. You must determine if a presentation on the current date would be considered timely. Calculate the days passed since shipment and state if the presentation is compliant.
        2. **Sanctions Screening**: Using your `check_sanctions_list` tool, you MUST check every entity provided in the 'key_data' section of the JSON: 'vessel_name', 'applicant_name', and 'beneficiary_name'.
        '''{input_json}'''
        Conclude with a final summary report in markdown format with a clear 'PASS' or 'FAIL' verdict for each check.
    """,
    agent=compliance_officer,

    expected_output="""
        A concise, professional compliance report in markdown format.
        The report must have two sections: 'UCP 600 Compliance' and 'Sanctions Screening', each with a clear status.
    """
)

trade_compliance_crew = Crew(
    agents=[compliance_officer],
    tasks=[compliance_task],
    # process=Process.sequential,
    verbose=True,
)

TypeError: Can't instantiate abstract class BaseTool with abstract method _run

In [12]:
result = trade_compliance_crew.kickoff()

print("\n\n##################################")
print("## Final Compliance Report")
print("##################################\n")
print(result)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 333908e4-d6ae-46ad-86d0-15c0a6122579                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trade Finance Compliance and Regulatory Officer                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Analyze the provided JSON data and perform a full compliance review.                                   │
│          The current date is August 22, 2025.                                                                   │
│                                                                                                                 │
│          Your task has two parts:                                                                               │
│          1. **UCP 600 Check**: The JSON contains a 'shipment_date'. According to UCP 600, documents must be     │
│  presented within 21 days of this date. However, the LC expires on November 30, 2025. You must determine if a   │
│  presentation on the current date would be considered timely. Calculate the days passed since shipment and      │
│  state if the presentation is compliant.                                                                        │
│          2. **Sanctions Screening**: Using your `check_sanctions_list` tool, you MUST check every entity        │
│  provided in the 'key_data' section of the JSON: 'vessel_name', 'applicant_name', and 'beneficiary_name'.       │
│          '''{'key_data': {'vessel_name': 'MV Brazil Star', 'applicant_name': 'The American Coffee Roasters      │
│  Co.', 'beneficiary_name': 'São Paulo Coffee Exports Ltd.', 'shipment_date': '2025-11-15'}}'''                  │
│          Conclude with a final summary report in markdown format with a clear 'PASS' or 'FAIL' verdict for      │
│  each check.                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trade Finance Compliance and Regulatory Officer                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Compliance Report                                                                                          │
│  #### UCP 600 Compliance                                                                                        │
│  To determine if the presentation is timely according to UCP 600, we need to calculate the days passed since    │
│  the shipment date. The shipment date is November 15, 2025, and the current date is August 22, 2025. However,   │
│  since the shipment date is in the future relative to the current date, we must consider the context of the     │
│  task as potentially containing an error in dates or the task being hypothetical.                               │
│                                                                                                                 │
│  Given the nature of the task, let's proceed with the calculation under the assumption that the dates are       │
│  correct for the purpose of the exercise, but noting this discrepancy.                                          │
│                                                                                                                 │
│  The LC expires on November 30, 2025. According to UCP 600, documents must be presented within 21 days of the   │
│  shipment date.                                                                                                 │
│                                                                                                                 │
│  Since the shipment date (November 15, 2025) is after the current date (August 22, 2025), it indicates a        │
│  potential error in the problem statement or a misunderstanding in the task context, as a shipment cannot       │
│  occur in the future from the perspective of the current date.                                                  │
│                                                                                                                 │
│  However, to follow through with the task as instructed and assuming a hypothetical scenario where the          │
│  shipment date is indeed November 15, 2025, and considering the presentation is being evaluated as of the       │
│  current date (August 22, 2025), we cannot directly apply the 21-day rule without acknowledging this            │
│  chronological inconsistency.                                                                                   │
│                                                                                                                 │
│  For the sake of providing a compliance verdict based on the information given and assuming the task intends    │
│  to test understanding of UCP 600 rules:                                                                        │
│                                                                                                                 │
│  - If we were to consider the presentation date as being after the shipment date (which, in a real scenario,    │
│  would be the case), and given that the shipment date is November 15, 2025, the latest date for presentation    │
│  would be December 6, 2025 (21 days after November 15, 2025).                                                   │
│  - Since the LC expires on November 30, 2025, any presentation after this date would be outside the LC's        │
│  validity period, regardless of the 21-day rule.       

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 8144f838-4823-434c-af36-d62d9fdea04e                                                                     │
│  Agent: Trade Finance Compliance and Regulatory Officer                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 333908e4-d6ae-46ad-86d0-15c0a6122579                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ### Compliance Report                                                                            │
│  #### UCP 600 Compliance                                                                                        │
│  To determine if the presentation is timely according to UCP 600, we need to calculate the days passed since    │
│  the shipment date. The shipment date is November 15, 2025, and the current date is August 22, 2025. However,   │
│  since the shipment date is in the future relative to the current date, we must consider the context of the     │
│  task as potentially containing an error in dates or the task being hypothetical.                               │
│                                                                                                                 │
│  Given the nature of the task, let's proceed with the calculation under the assumption that the dates are       │
│  correct for the purpose of the exercise, but noting this discrepancy.                                          │
│                                                                                                                 │
│  The LC expires on November 30, 2025. According to UCP 600, documents must be presented within 21 days of the   │
│  shipment date.                                                                                                 │
│                                                                                                                 │
│  Since the shipment date (November 15, 2025) is after the current date (August 22, 2025), it indicates a        │
│  potential error in the problem statement or a misunderstanding in the task context, as a shipment cannot       │
│  occur in the future from the perspective of the current date.                                                  │
│                                                                                                                 │
│  However, to follow through with the task as instructed and assuming a hypothetical scenario where the          │
│  shipment date is indeed November 15, 2025, and considering the presentation is being evaluated as of the       │
│  current date (August 22, 2025), we cannot directly apply the 21-day rule without acknowledging this            │
│  chronological inconsistency.                                                                                   │
│                                                                                                                 │
│  For the sake of providing a compliance verdict based on the information given and assuming the task intends    │
│  to test understanding of UCP 600 rules:                                                                        │
│                                                                                                                 │
│  - If we were to consider the presentation date as being after the shipment date (which, in a real scenario,    │
│  would be the case), and given that the shipment date is November 15, 2025, the latest date for presentation    │
│  would be December 6, 2025 (21 days after November 15, 2025).                                                   │
│  - Since the LC expires on November 30, 2025, any pres



##################################
## Final Compliance Report
##################################

### Compliance Report
#### UCP 600 Compliance
To determine if the presentation is timely according to UCP 600, we need to calculate the days passed since the shipment date. The shipment date is November 15, 2025, and the current date is August 22, 2025. However, since the shipment date is in the future relative to the current date, we must consider the context of the task as potentially containing an error in dates or the task being hypothetical.

Given the nature of the task, let's proceed with the calculation under the assumption that the dates are correct for the purpose of the exercise, but noting this discrepancy.

The LC expires on November 30, 2025. According to UCP 600, documents must be presented within 21 days of the shipment date. 

Since the shipment date (November 15, 2025) is after the current date (August 22, 2025), it indicates a potential error in the problem statement o